# 05 — Collaborative Filtering

**Goal:** recommend movies using patterns in user ratings. Missing ratings remain missing; they are never treated as zero.

## 1. Load ratings and movie catalogue

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Keep paths consistent whether the notebook is launched from the project root or 02_Notebooks.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "02_Notebooks" else CWD
DATA_RAW = PROJECT_ROOT / "01_Data" / "Raw"
DATA_PROCESSED = PROJECT_ROOT / "01_Data" / "Processed"
MODELS = PROJECT_ROOT / "03_Models"
VISUALS = PROJECT_ROOT / "05_Visualizations"
for folder in [DATA_RAW, DATA_PROCESSED, MODELS, VISUALS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import joblib
movies = pd.read_pickle(DATA_PROCESSED / "movies_processed.pkl").reset_index(drop=True)
ratings = pd.read_csv(DATA_PROCESSED / "ratings_processed.csv")
print("Ratings:", ratings.shape, "| users:", ratings.userId.nunique(), "| movies rated:", ratings.movieId.nunique())

Project root: D:\DataScience\notebooks\CineMatch_Ready_Notebooks
Ratings: (100836, 4) | users: 610 | movies rated: 9724


## 2. Create a sparse user-movie matrix

Rows represent users and columns represent movies. Only observed ratings are stored in the sparse matrix; an absent entry means the user did not rate that movie.

In [2]:
user_ids = np.sort(ratings["userId"].unique())
rated_movie_ids = np.sort(ratings["movieId"].unique())
user_to_idx = {int(uid): i for i, uid in enumerate(user_ids)}
movie_to_idx = {int(mid): i for i, mid in enumerate(rated_movie_ids)}
rows = ratings["userId"].map(user_to_idx).to_numpy()
cols = ratings["movieId"].map(movie_to_idx).to_numpy()
values = ratings["rating"].astype(float).to_numpy()
user_item = csr_matrix((values, (rows, cols)), shape=(len(user_ids), len(rated_movie_ids)))
print("Sparse matrix shape:", user_item.shape)
print("Observed interactions:", user_item.nnz)
print("Density:", round(user_item.nnz / (user_item.shape[0]*user_item.shape[1]), 5))

Sparse matrix shape: (610, 9724)
Observed interactions: 100836
Density: 0.017


## 3. Item-item similarity

Item-based collaborative filtering compares rating patterns across users. We centre each user's ratings around their own mean before computing cosine similarity, reducing the effect of users who consistently rate high or low.

In [3]:
user_means = np.asarray(user_item.sum(axis=1)).ravel() / np.maximum(np.diff(user_item.indptr), 1)
centered = user_item.copy().astype(float)
for user_idx in range(centered.shape[0]):
    start, end = centered.indptr[user_idx], centered.indptr[user_idx+1]
    centered.data[start:end] -= user_means[user_idx]

# Similarity is computed for movie columns; sparse output keeps storage manageable.
item_similarity = cosine_similarity(centered.T, dense_output=False).tocsr()
item_similarity.setdiag(0)
item_similarity.eliminate_zeros()
print("Item similarity matrix:", item_similarity.shape, "| nonzero similarities:", item_similarity.nnz)

Item similarity matrix: (9724, 9724) | nonzero similarities: 26314686


## 4. User-item recommendation function

For each unseen movie, aggregate the user's ratings weighted by similarity to that movie. Only movies with positive evidence are returned.

In [4]:
def recommend_for_user(user_id, top_n=10, min_similarity=0.0):
    if int(user_id) not in user_to_idx:
        return pd.DataFrame(columns=["movieId","title","predicted_score"])
    uidx = user_to_idx[int(user_id)]
    start, end = user_item.indptr[uidx], user_item.indptr[uidx+1]
    seen_cols = user_item.indices[start:end]
    seen_ratings = user_item.data[start:end]
    if len(seen_cols) == 0:
        return pd.DataFrame(columns=["movieId","title","predicted_score"])

    scores = np.zeros(len(rated_movie_ids), dtype=float)
    weights = np.zeros(len(rated_movie_ids), dtype=float)
    for movie_col, rating_value in zip(seen_cols, seen_ratings):
        similarities = item_similarity[:, movie_col].toarray().ravel()
        valid = similarities > min_similarity
        scores[valid] += similarities[valid] * rating_value
        weights[valid] += np.abs(similarities[valid])
    predicted = np.divide(scores, weights, out=np.zeros_like(scores), where=weights > 0)
    predicted[seen_cols] = -np.inf
    best = np.argsort(predicted)[::-1]
    output = []
    for col in best:
        if not np.isfinite(predicted[col]) or weights[col] == 0:
            continue
        mid = int(rated_movie_ids[col])
        output.append({"movieId": mid, "predicted_score": float(predicted[col])})
        if len(output) >= top_n:
            break
    result = pd.DataFrame(output).merge(movies[["movieId","title","genres","average_rating"]], on="movieId", how="left")
    return result

## 5. Test with an actual user ID

The first available user ID is used only as a reproducible example. You can replace it with any user ID printed below.

In [5]:
sample_user_id = int(user_ids[0])
print("Testing user:", sample_user_id)
display(recommend_for_user(sample_user_id, top_n=10))

Testing user: 1


,movieId,predicted_score,title,genres,average_rating
0,167732,5.0,A Street Cat Named Bob (2016),Comedy|Drama,3.5
1,2695,5.0,"Boys, The (1998)",Drama,2.0
2,1739,5.0,3 Ninjas: High Noon On Mega Mountain (1998),Action|Children|Comedy,4.0
3,141718,5.0,Deathgasm (2015),Comedy|Horror,5.0
4,1477,5.0,Love Jones (1997),Romance,4.0
5,4505,5.0,For Keeps (1988),Drama|Romance,3.0
6,168712,5.0,Fifty Shades Darker (2017),Drama|Romance,1.0
7,88746,5.0,Terri (2011),Comedy,3.5
8,99853,5.0,Codependent Lesbian Space Alien Seeks Same (2011),Comedy|Romance|Sci-Fi,4.0
9,118834,5.0,National Lampoon's Bag Boy (2007),Comedy,5.0


## 6. User-based similarity concept

User-based filtering compares users with similar rating patterns. It can be expensive for large datasets; here we show how to calculate a sparse similarity matrix for learning purposes. The main recommender above uses item-item similarity.

In [6]:
# User similarity on the same centred sparse interaction matrix.
user_similarity = cosine_similarity(centered, dense_output=False).tocsr()
user_similarity.setdiag(0)
user_similarity.eliminate_zeros()
print("User similarity matrix:", user_similarity.shape)

User similarity matrix: (610, 610)


## 7. Save collaborative artifacts

In [7]:
collab_artifacts = {
    "user_item": user_item, "centered_user_item": centered,
    "item_similarity": item_similarity, "user_similarity": user_similarity,
    "user_ids": user_ids, "rated_movie_ids": rated_movie_ids,
    "user_to_idx": user_to_idx, "movie_to_idx": movie_to_idx
}
joblib.dump(collab_artifacts, MODELS / "collaborative_artifacts.joblib", compress=3)
print("Saved:", MODELS / "collaborative_artifacts.joblib")

Saved: D:\DataScience\notebooks\CineMatch_Ready_Notebooks\03_Models\collaborative_artifacts.joblib


## Concept comparison

- **Content-based:** compares movie attributes; it can recommend an unrated/new movie if metadata exists.
- **Collaborative:** learns from user-item interaction patterns; it can discover taste patterns but struggles with new users/items and sparse ratings.

Next: **06_Hybrid_Recommendation_System.ipynb**.